# DocuMind – RAG experiments
Run from the project root (`jupyter lab`). Uses the same modules as the app.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dataclasses import replace
from config.settings import load_settings
settings = load_settings()
settings

## 1. Load and chunk a sample PDF

In [ ]:
from src.document_loader import load_pdf
from src.text_splitter import split_pages
res = load_pdf('../data/samples/Artificial_Intelligence_Basics.pdf', filename='sample.pdf', doc_id='nb')
chunks = split_pages(res.pages, settings.chunk_size, settings.chunk_overlap)
len(res.pages), len(chunks), chunks[0].metadata

## 2. Index in an experiment-only Chroma folder and retrieve
Try different `chunk_size` / `top_k` values and compare the scores.

In [ ]:
import tempfile
from src.embeddings import get_embeddings
from src.vector_store import VectorStore
exp = replace(settings, chroma_dir=__import__('pathlib').Path(tempfile.mkdtemp()))
vs = VectorStore(exp, get_embeddings(exp), 'notebook-user')
vs.add_chunks(chunks, 'nb')
from src.retriever import retrieve
for h in retrieve(vs, 'What is machine learning?', k=4, doc_ids=['nb']):
    print(round(h.score, 3), h.filename, 'p.', h.page, '|', h.text[:80].replace('\n', ' '))

## 3. Ask a question (needs an LLM configured in `.env`)

In [ ]:
from src.llm import get_llm
from src.rag_pipeline import RAGPipeline
ans = RAGPipeline(vs, get_llm(settings), top_k=4).ask('What are the benefits of AI?', ['nb'])
print(ans.text)
[(s.filename, s.page) for s in ans.sources]